# Visual Question Answering (VQA v2) Baseline
## ???????????? ?????????? ??? ??????? ?????? (IMRAD)

???? ??????? ????????? ?????? ???????? ????????????:
1. ????????? ????? ? ????????? VQA v2 ? MS COCO 2014 ? `/kaggle/input`.
2. ??????-?????????? ????????? ???????? ????? ResNet-50 (2048-dim avgpool) ? `.h5`.
3. ???????? ??????????????? baseline-?????? (ResNet-50 + LSTM ? `pack_padded_sequence` + Hadamard/Concat Fusion).
4. ???????? ??????????? ??????????? ?????? (Question-Only Baseline) ??? ???????? ???????? ?????????????????.
5. ?????? ???????? (VQA Accuracy ?? ??????????), ?????????? ???????? ???????? ? ??????? `predictions.csv` ??? PostgreSQL.

### ?????? 1: ????????? ??????? ?????? ? /kaggle/input

In [ ]:
import os
from pathlib import Path

print("=== ???????? ???????????? ????????? ? /kaggle/input ===")
base_path = Path("/kaggle/input")
if base_path.exists():
    for root, dirs, files in os.walk(base_path):
        depth = len(Path(root).relative_to(base_path).parts)
        if depth <= 2:
            print(f"{'  ' * depth}?? {Path(root).name}/ ({len(files)} ??????, {len(dirs)} ?????)")
            for f in files[:3]:
                print(f"{'  ' * depth}   ?? {f}")
else:
    print("?????????? /kaggle/input ?? ??????????. ?????? ?????????? ?? ?? Kaggle.")

### ?????? 2: ???????????? ??????????? ??????? ? ????????? ????????????

In [ ]:
# ????????? ??? ??????????? ? ??????? ????? Kaggle
!rm -rf /kaggle/working/science
!git clone https://github.com/TryHanger/science.git /kaggle/working/science
%cd /kaggle/working/science

import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

!pip install -q -r requirements.txt
print("? ??????????? ??????? ?????????? ? ????? ? ??????:", os.getcwd())

### ?????? 3: ??? 1 ? ??????-?????????? ?????????? ????????? (ResNet-50 -> HDF5)

In [ ]:
!python src/extract_features.py --env kaggle

### ?????? 4: ??? 4 ? ???????? ??????????????? baseline-?????? (ResNet + LSTM)

In [ ]:
!python src/train.py --env kaggle --model-type vqa

### ?????? 5: ??? 4 (????????) ? ???????? Question-Only ?????? (Ablation Study ??? IMRAD)

In [ ]:
!python src/train.py --env kaggle --model-type question_only

### ?????? 6: ??? 5 ? ?????? (Results): ??????? ??????, ???????, ??????? ? CSV ??? PostgreSQL

In [ ]:
!python src/evaluate.py --env kaggle

### ?????? 7: ??? 6 ? ????????????? ???????? ??????? predict(image_path, question)

In [ ]:
from src.predict import predict
import glob

# ????? ????????? ???????? ???????? ?? val2014
test_images = glob.glob("/kaggle/input/coco2014/val2014/*.jpg")
if test_images:
    sample_img = test_images[0]
    ans, conf = predict(sample_img, "is the picture colorful?", env="kaggle")
    print(f"????????: {sample_img}")
    print(f"????? ??????: {ans} (???????????: {conf:.2%})")
else:
    print("??????? ?????? ???? ? ???????? ??? ?????.")